In [157]:
from pymongo import MongoClient
import re
import os
from typing import List, Dict, Any
import shutil

In [158]:
def connect_mongodb(uri,db_name, collection_name):
    # Connect to MongoDB 
    client = MongoClient(uri)
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]
    
    return collection
def get_all_directories(root_path):
    """
    Get all directory names under the given root path.
    Returns a list of directory names.
    """
    return [name for name in os.listdir(root_path) if os.path.isdir(os.path.join(root_path, name))]
def contains_original(lst):
    target = "original"
    return any(item.lower() == target for item in lst)
def add_data_to_mongodb(collection, app_id, level_2_path_list):
    # Tạo document
    document = {
        "_id": app_id,
        "level_2_path": level_2_path_list
    }
    try:
        collection.insert_one(document)
        print("Data inserted successfully!")
    except Exception as e:
        print("Error inserting data:", e)
def get_all_ids(collection):
    return [doc['_id'] for doc in collection.find({}, {'_id': 1})]
def get_document_by_id(collection, target_id):
    return collection.find_one({"_id": target_id})
def convert_level_2_to_object_and_add_level_3(collection, app_id, level_2_value, level_3_values):
    # Read the existing document
    doc = collection.find_one({"_id": app_id})
    if not doc:
        print("Document not found.")
        return

    new_level_2_path = []
    for path in doc.get("level_2_path", []):
        if path == level_2_value:
            new_level_2_path.append({
                "path": path,
                "level_3_path": level_3_values
            })
        else:
            new_level_2_path.append(path)

    # Update the document
    result = collection.update_one(
        {"_id": app_id},
        {"$set": {"level_2_path": new_level_2_path}}
    )

    if result.modified_count > 0:
        print("Update successful!")
    else:
        print("No changes were made.")


In [159]:
mongoDB_uri = 'mongodb://localhost:27017'
mongoDB_database = 'wearable-project-2' 
mongoDB_collection = 'codeblock_ast'

In [160]:
# Connect to the MongoDB collection
collection = connect_mongodb(mongoDB_uri,mongoDB_database,mongoDB_collection)

In [161]:
root_path = r"E:\wearable-2-dataset-code-extraction"

In [162]:
array_package_name = get_all_ids(collection)
for x in range(len(array_package_name)):
    print("------------ Loop-"+str(x)+" ------------")
    package_name = array_package_name[x]
    print("package name: ",package_name)
    data = get_document_by_id(collection, package_name)
    print("data: ",data)
    level_2_path_array = data["level_2_path"]
    print("level_2_path_array: ",level_2_path_array)
    for y in range(len(level_2_path_array)):
        level_2_path = level_2_path_array[y]
        level_3_directory_array = get_all_directories(level_2_path)
        print(level_3_directory_array)
        level_3_path_array = []
        for z in range(len(level_3_directory_array)):
            level_3_directory = level_3_directory_array[z]
            level_3_path = level_2_path+"\\"+level_3_directory
            level_3_path_array.append(level_3_path)
        print(level_3_path_array)
        convert_level_2_to_object_and_add_level_3(collection, package_name, level_2_path, level_3_path_array)
    # break

------------ Loop-0 ------------
package name:  adarshurs.android.vlcmobileremote
data:  {'_id': 'adarshurs.android.vlcmobileremote', 'level_2_path': ['E:\\wearable-2-dataset-code-extraction\\adarshurs.android.vlcmobileremote\\Google']}
level_2_path_array:  ['E:\\wearable-2-dataset-code-extraction\\adarshurs.android.vlcmobileremote\\Google']
['com.google.android.gms.ads.MobileAds', 'com.google.firebase.analytics.connector.AnalyticsConnector', 'com.google.firebase.components.ComponentRegistrar', 'com.google.firebase.provider.FirebaseInitProvider']
['E:\\wearable-2-dataset-code-extraction\\adarshurs.android.vlcmobileremote\\Google\\com.google.android.gms.ads.MobileAds', 'E:\\wearable-2-dataset-code-extraction\\adarshurs.android.vlcmobileremote\\Google\\com.google.firebase.analytics.connector.AnalyticsConnector', 'E:\\wearable-2-dataset-code-extraction\\adarshurs.android.vlcmobileremote\\Google\\com.google.firebase.components.ComponentRegistrar', 'E:\\wearable-2-dataset-code-extraction\\a